In [105]:
import pandas as pd
import numpy as np

## Data Transform

### Categorical feature

In [106]:
X_train_df_cleaned = pd.read_csv('dataset/processed/train_processed.csv')
X_test_df_cleaned = pd.read_csv('dataset/processed/test_processed.csv')

In [107]:
X_train_cleaned = X_train_df_cleaned.drop(columns=['Severity'])
y_train_cleaned = X_train_df_cleaned['Severity']
    
X_test_cleaned = X_test_df_cleaned.drop(columns=['Severity'])
y_test_cleaned = X_test_df_cleaned['Severity']

In [108]:
X_train_cleaned.columns

Index(['Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Amenity', 'City',
       'County', 'Crossing', 'Junction', 'Railway', 'Start_Time', 'State',
       'Station', 'Stop', 'Sunrise_Sunset', 'Traffic_Signal',
       'Weather_Condition'],
      dtype='object')

In [109]:
X_train_cleaned['Start_Time'] = pd.to_datetime(X_train_cleaned['Start_Time'], errors='coerce')
X_test_cleaned['Start_Time'] = pd.to_datetime(X_test_cleaned['Start_Time'], errors='coerce')

In [110]:
bool_col = ['Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal']

for col in bool_col:
    X_train_cleaned[col] = X_train_cleaned[col].astype('object')
    X_test_cleaned[col] = X_test_cleaned[col].astype('object')

In [111]:
X_train_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2492096 entries, 0 to 2492095
Data columns (total 16 columns):
 #   Column             Dtype         
---  ------             -----         
 0   Temperature(F)     float64       
 1   Humidity(%)        float64       
 2   Visibility(mi)     float64       
 3   Amenity            object        
 4   City               object        
 5   County             object        
 6   Crossing           object        
 7   Junction           object        
 8   Railway            object        
 9   Start_Time         datetime64[ns]
 10  State              object        
 11  Station            object        
 12  Stop               object        
 13  Sunrise_Sunset     object        
 14  Traffic_Signal     object        
 15  Weather_Condition  object        
dtypes: datetime64[ns](1), float64(3), object(12)
memory usage: 304.2+ MB


In [112]:
categorical_features = X_train_cleaned.select_dtypes(include=['object'])
categorical_features

,Amenity,City,County,Crossing,Junction,Railway,State,Station,Stop,Sunrise_Sunset,Traffic_Signal,Weather_Condition
0,False,Hillsborough,Orange,False,False,False,NC,False,False,Night,False,NaN
1,False,Auburn,Placer,False,False,False,CA,False,True,Day,False,Cloudy
2,False,Richmond,Richmond City,True,False,False,VA,False,False,Day,True,Fair
3,False,Newport News,Newport News,False,True,False,VA,False,False,Night,False,Fair
4,False,Sacramento,Sacramento,False,True,False,CA,False,False,Day,False,Fair
...,...,...,...,...,...,...,...,...,...,...,...,...
2492091,True,Miami,Miami-Dade,True,False,False,FL,True,False,Day,True,Fair
2492092,False,Fairfax,Allendale,False,False,False,SC,False,False,Day,False,Fair
2492093,False,Statesville,Iredell,False,False,False,NC,False,False,Day,False,Fair
2492094,False,Dallas,Dallas,True,False,False,TX,False,False,Day,True,Light Drizzle


In [113]:
#  categorical feature
for col in categorical_features.columns:
    unique_vals = X_train_cleaned[col].unique()
    n_unique = len(unique_vals)
    
    print(f"\n📌 Feature: {col}")
    print(f"   • Total unique values: {n_unique}")
    print(f"   • Missing values: {X_train_cleaned[col].isnull().sum()} ({X_train_cleaned[col].isnull().sum()/len(X_train_cleaned)*100:.2f}%)")
    
    # show unique values
    if n_unique <= 50:
        print(f"   • Unique values: {sorted([str(v) for v in unique_vals if pd.notna(v)])}")
    else:
        print(f"   • Too many unique values ({n_unique}). Showing top 20:")
        value_counts = X_train_cleaned[col].value_counts()
        for val, count in value_counts.head(20).items():
            pct = (count / len(X_train_cleaned)) * 100
            print(f"      {str(val):<40} : {count:>8,} ({pct:>6.2f}%)")
    
    print("-" * 100)

print(f"\n📊 Summary: {len(categorical_features.columns)} categorical features in total")
print("=" * 100)


📌 Feature: Amenity
   • Total unique values: 2
   • Missing values: 0 (0.00%)
   • Unique values: ['False', 'True']
----------------------------------------------------------------------------------------------------

📌 Feature: City
   • Total unique values: 11521
   • Missing values: 0 (0.00%)
   • Too many unique values (11521). Showing top 20:

📌 Feature: City
   • Total unique values: 11521
   • Missing values: 0 (0.00%)
   • Too many unique values (11521). Showing top 20:
      Miami                                    :   92,617 (  3.72%)
      Los Angeles                              :   54,732 (  2.20%)
      Orlando                                  :   47,711 (  1.91%)
      Dallas                                   :   32,738 (  1.31%)
      Houston                                  :   30,519 (  1.22%)
      Sacramento                               :   26,598 (  1.07%)
      Charlotte                                :   25,946 (  1.04%)
      San Diego                         

### CREATE AGGREGATION FEATURES (CITY / COUNTY)

In [114]:
def add_accident_density(df):
    city_counts = df.groupby('City').size().reset_index(name='Accidents_City')
    county_counts = df.groupby('County').size().reset_index(name='Accidents_County')

    df = df.merge(city_counts, on='City', how='left')
    df = df.merge(county_counts, on='County', how='left')

    return df

X_train_cleaned = add_accident_density(X_train_cleaned)
X_test_cleaned = add_accident_density(X_test_cleaned)

### Encoding Strategy

**Approach:**
- **Boolean features** (7 features): Label Encoding (False=0, True=1)
- **State** (49 unique): Target Encoding (mean Severity per state)
- **City** (11,521 unique): Target Encoding (mean Severity per city)
- **County** (1,745 unique): Target Encoding (mean Severity per county)

Target Encoding is used for high cardinality features to avoid curse of dimensionality while capturing the relationship with target variable.

In [115]:
from sklearn.preprocessing import LabelEncoder

def encode_boolean_features(X_train, X_test, bool_features):
    le = LabelEncoder()
    
    for col in bool_features:
        X_train[col] = le.fit_transform(X_train[col])
        X_test[col] = le.transform(X_test[col])
    
    print("✅ Boolean features encoded (False=0, True=1)")
    print(f"Encoded features: {bool_features}")
    
    return X_train, X_test

In [116]:
def encode_target_features(X_train, X_test, y_train, high_cardinality_features):
    """
    Apply Target Encoding to high cardinality categorical features.
    
    Parameters:
    -----------
    X_train : pd.DataFrame
        Training dataset
    X_test : pd.DataFrame
        Test dataset
    y_train : pd.Series
        Target variable for training set
    high_cardinality_features : list
        List of high cardinality feature names to encode
    
    Returns:
    --------
    X_train, X_test, target_encoding_map : tuple
        Encoded datasets and mapping dictionary
    """
    # Store encoding mappings
    target_encoding_map = {}
    global_mean = y_train.mean()
    
    for col in high_cardinality_features:
        # Calculate mean target for each category
        # Group by category and calculate mean of corresponding target values
        encoding = X_train.groupby(col).apply(
            lambda x: y_train.loc[x.index].mean()
        )
        target_encoding_map[col] = encoding
        
        # Apply encoding
        X_train[col + '_encoded'] = X_train[col].map(encoding)
        X_test[col + '_encoded'] = X_test[col].map(encoding)
        
        # Fill missing values (categories not in train) with global mean
        X_train[col + '_encoded'] = X_train[col + '_encoded'].fillna(global_mean)
        X_test[col + '_encoded'] = X_test[col + '_encoded'].fillna(global_mean)
        
        # Drop original column
        X_train = X_train.drop(columns=[col])
        X_test = X_test.drop(columns=[col])
        
        print(f"✅ {col} encoded using Target Encoding")
    
    print(f"\n📊 Total features after encoding: {X_train.shape[1]}")
    
    return X_train, X_test, target_encoding_map


#### Why These Encoding Methods?

**1. Label Encoding for Boolean Features (7 features)**
- **Applied to:** Amenity, Crossing, Junction, Railway, Station, Stop, Traffic_Signal
- **Rationale:** These are binary features (True/False) with natural ordering. Simple mapping (False=0, True=1) preserves the binary nature without adding dimensionality.
- **Why not One-Hot?** Unnecessary for binary features - would create redundant columns.

**2. Target Encoding for High Cardinality Features**
- **Applied to:** City (11,521 unique), County (1,745 unique), State (49 unique)
- **Rationale:** 
  - **Avoids Curse of Dimensionality:** One-hot encoding City would create 11,521 columns, leading to sparse matrices and overfitting
  - **Captures Target Relationship:** Encodes categories based on their mean Severity, preserving predictive information
  - **Handles Unseen Categories:** Missing categories in test set are filled with global mean
  
**Why not other methods?**
- **One-Hot Encoding:** Creates 13,315 columns (11,521 + 1,745 + 49), causing memory issues and overfitting
- **Frequency Encoding:** Only captures occurrence frequency, not the relationship with accident severity
- **Hash Encoding:** May cause collisions and loses interpretability

**Result:** Efficient encoding that maintains predictive power while keeping dimensionality manageable.

In [117]:
# Apply encoding
bool_features = ['Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal']
X_train_cleaned, X_test_cleaned = encode_boolean_features(X_train_cleaned, X_test_cleaned, bool_features)

✅ Boolean features encoded (False=0, True=1)
Encoded features: ['Amenity', 'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal']


In [118]:
# Apply Target Encoding
high_cardinality_features = ['City', 'County', 'State']
X_train_cleaned, X_test_cleaned, target_encoding_map = encode_target_features(
    X_train_cleaned, X_test_cleaned, y_train_cleaned, high_cardinality_features
)

C:\Users\dangn\AppData\Local\Temp\ipykernel_23860\1957745699.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  encoding = X_train.groupby(col).apply(


✅ City encoded using Target Encoding


C:\Users\dangn\AppData\Local\Temp\ipykernel_23860\1957745699.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  encoding = X_train.groupby(col).apply(


✅ County encoded using Target Encoding


C:\Users\dangn\AppData\Local\Temp\ipykernel_23860\1957745699.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  encoding = X_train.groupby(col).apply(


✅ State encoded using Target Encoding

📊 Total features after encoding: 18


In [119]:
# Verify encoding results
print("🔍 Data info after encoding:")
print(f"Train shape: {X_train_cleaned.shape}")
print(f"Test shape: {X_test_cleaned.shape}")
print(f"\nData types:")
print(X_train_cleaned.dtypes)
print(f"\nFirst 5 rows:")
X_train_cleaned.head()

🔍 Data info after encoding:
Train shape: (2492096, 18)
Test shape: (623050, 18)

Data types:
Temperature(F)              float64
Humidity(%)                 float64
Visibility(mi)              float64
Amenity                       int64
Crossing                      int64
Junction                      int64
Railway                       int64
Start_Time           datetime64[ns]
Station                       int64
Stop                          int64
Sunrise_Sunset               object
Traffic_Signal                int64
Weather_Condition            object
Accidents_City                int64
Accidents_County              int64
City_encoded                float64
County_encoded              float64
State_encoded               float64
dtype: object

First 5 rows:


,Temperature(F),Humidity(%),Visibility(mi),Amenity,Crossing,Junction,Railway,Start_Time,Station,Stop,Sunrise_Sunset,Traffic_Signal,Weather_Condition,Accidents_City,Accidents_County,City_encoded,County_encoded,State_encoded
0,63.0,66.0,10.0,0,0,0,0,2022-03-18 06:55:00,0,0,Night,0,NaN,305,98012,2.288525,2.028741,2.126316
1,41.0,93.0,10.0,0,0,0,0,2023-01-29 16:35:00,0,1,Day,0,Cloudy,2022,7649,2.067755,2.017519,2.019478
2,71.0,32.0,10.0,0,1,0,0,2022-10-11 12:53:40,0,0,Day,1,Fair,15223,7050,1.992643,1.954752,2.149949
3,26.0,99.0,10.0,0,0,1,0,2022-01-24 06:12:00,0,0,Night,0,Fair,1673,1173,2.107591,2.110827,2.149949
4,64.0,52.0,10.0,0,0,1,0,2020-02-14 15:28:00,0,0,Day,0,Fair,26598,38767,2.000564,2.003018,2.019478


### Feature Engineering

Based on the current features, we'll create new meaningful features to improve model performance:

**1. Temporal Features from Start_Time:**
- Hour of day (rush hour patterns)
- Day of week (weekend vs weekday)
- Month (seasonal patterns)
- Is weekend (binary)
- Time period (Morning/Afternoon/Evening/Night)

**2. Weather-Related Features:**
- Temperature categories (Cold/Mild/Hot)
- Visibility categories (Poor/Moderate/Good)
- Humidity categories (Low/Medium/High)
- Weather severity score (combining multiple weather factors)

**3. Location-Based Features:**
- Road infrastructure score (sum of Amenity, Crossing, Junction, Railway, Station, Stop, Traffic_Signal)
- Is complex intersection (Junction + Traffic_Signal)

**4. Interaction Features:**
- Weather + Time interactions (weather severity during night/day)
- Infrastructure + Time interactions (complex intersections during rush hour)

#### 1. TIME-BASED FEATURES

In [120]:
def add_time_features(df):
    df['Start_Time'] = pd.to_datetime(df['Start_Time'])
    df['Hour'] = df['Start_Time'].dt.hour
    df['DayOfWeek'] = df['Start_Time'].dt.dayofweek
    df['Is_Weekend'] = df['DayOfWeek'].isin([5, 6, 7]).astype(int)
    df['Month'] = df['Start_Time'].dt.month
    
    # Season encoded as integers (1=Winter, 2=Spring, 3=Summer, 4=Fall)
    def get_season(m):
        if m in [12, 1, 2]: return 1
        if m in [3, 4, 5]:  return 2
        if m in [6, 7, 8]:  return 3
        return 4
    
    def rush_hour(h):
        return 1 if (7 <= h <= 9) or (17 <= h <= 19) else 0

    df['Is_Rush_Hour'] = df['Hour'].apply(rush_hour)
    df['Season'] = df['Month'].apply(get_season)
    drop_columns = ['Month', 'DayOfWeek', 'Start_Time', 'Hour']
    df = df.drop(columns=drop_columns)
    return df

#### 2. WEATHER FEATURES

In [121]:
def add_weather_flags(df):
    df['Weather_Condition'] = df['Weather_Condition'].fillna('').str.lower()

    df['Is_Rain']   = df['Weather_Condition'].str.contains('rain|storm|shower').astype(int)
    df['Is_Snow']   = df['Weather_Condition'].str.contains('snow|blizzard|sleet').astype(int)
    df['Is_Fog']    = df['Weather_Condition'].str.contains('fog|mist|haze').astype(int)
    df['Is_Clear']  = df['Weather_Condition'].str.contains('clear').astype(int)
    df['Is_Cloudy'] = df['Weather_Condition'].str.contains('cloud').astype(int)

    df['Low_Visibility'] = (df['Visibility(mi)'] < 3).astype(int)
    df['Low_Temp']       = (df['Temperature(F)'] < 40).astype(int)
    df['High_Humidity']  = (df['Humidity(%)'] > 90).astype(int)

    df['Bad_Weather'] = (
        df['Is_Rain'] |
        df['Is_Snow'] |
        df['Is_Fog'] |
        df['Low_Visibility'] |
        df['High_Humidity']
    ).astype(int)

    drop_columns = ['Weather_Condition']
    df = df.drop(columns=drop_columns)
    return df

#### 3. ROAD CONTEXT FEATURES

In [122]:
def add_road_context_score(df):
    poi_cols = [
        'Amenity', 'Crossing', 'Junction', 'Railway',
        'Station', 'Stop', 'Traffic_Signal'
    ]
    df['Road_Context_Score'] = df[poi_cols].sum(axis=1)
    return df

#### 4. DAY/NIGHT FEATURES

In [123]:
def add_day_night(df):
    df['Is_Night'] = (df['Sunrise_Sunset'] == 'Night').astype(int)

    drop_columns = ['Sunrise_Sunset']
    df = df.drop(columns=drop_columns)
    return df

#### 5. INTERACTION FEATURES

In [124]:
def add_interactions(df):
    df['Night_Rain_Junction'] = (
        df['Is_Night'] &
        df['Is_Rain'] &
        df['Junction']
    ).astype(int)
    return df

#### 6. MASTER PIPELINE — chạy toàn bộ feature engineering

In [125]:
def apply_all_features(df):
    df = add_time_features(df)
    df = add_weather_flags(df)
    df = add_road_context_score(df)
    df = add_day_night(df)
    df = add_interactions(df)
    return df

In [128]:
# Summary of all features after engineering
X_train_featured = apply_all_features(X_train_cleaned)
X_test_featured = apply_all_features(X_test_cleaned)

print("📊 Feature Engineering Summary:")
print(f"Train shape: {X_train_featured.shape}")
print(f"Test shape: {X_test_featured.shape}")
print(f"\nTotal features: {X_train_featured.shape[1]}")
print("\nNew features created:")
new_features = [col for col in X_train_featured.columns if col not in ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 
                'Amenity', 'Crossing', 'Junction', 'Railway', 'Start_Time', 'Station', 'Stop', 
                'Sunrise_Sunset', 'Traffic_Signal', 'Weather_Condition', 'City_encoded', 'County_encoded', 'State_encoded']]
for i, feat in enumerate(new_features, 1):
    print(f"{i:2d}. {feat}")

print(f"\n🔍 Preview of engineered features:")
X_train_featured.head()

📊 Feature Engineering Summary:
Train shape: (2492096, 30)
Test shape: (623050, 30)

Total features: 30

New features created:
 1. Accidents_City
 2. Accidents_County
 3. Is_Weekend
 4. Is_Rush_Hour
 5. Season
 6. Is_Rain
 7. Is_Snow
 8. Is_Fog
 9. Is_Clear
10. Is_Cloudy
11. Low_Visibility
12. Low_Temp
13. High_Humidity
14. Bad_Weather
15. Road_Context_Score
16. Is_Night
17. Night_Rain_Junction

🔍 Preview of engineered features:


,Temperature(F),Humidity(%),Visibility(mi),Amenity,Crossing,Junction,Railway,Station,Stop,Traffic_Signal,...,Is_Fog,Is_Clear,Is_Cloudy,Low_Visibility,Low_Temp,High_Humidity,Bad_Weather,Road_Context_Score,Is_Night,Night_Rain_Junction
0,63.0,66.0,10.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,41.0,93.0,10.0,0,0,0,0,0,1,0,...,0,0,1,0,0,1,1,1,0,0
2,71.0,32.0,10.0,0,1,0,0,0,0,1,...,0,0,0,0,0,0,0,2,0,0
3,26.0,99.0,10.0,0,0,1,0,0,0,0,...,0,0,0,0,1,1,1,1,1,0
4,64.0,52.0,10.0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [130]:
X_train_featured.columns

Index(['Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Amenity',
       'Crossing', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal',
       'Accidents_City', 'Accidents_County', 'City_encoded', 'County_encoded',
       'State_encoded', 'Is_Weekend', 'Is_Rush_Hour', 'Season', 'Is_Rain',
       'Is_Snow', 'Is_Fog', 'Is_Clear', 'Is_Cloudy', 'Low_Visibility',
       'Low_Temp', 'High_Humidity', 'Bad_Weather', 'Road_Context_Score',
       'Is_Night', 'Night_Rain_Junction'],
      dtype='object')

### Save Data

In [131]:
# Save final feature-engineered datasets
X_train_featured.to_csv('dataset/processed/X_train_featured.csv', index=False)
X_test_featured.to_csv('dataset/processed/X_test_featured.csv', index=False)

# Also save target variables
y_train_cleaned.to_csv('dataset/processed/y_train.csv', index=False)
y_test_cleaned.to_csv('dataset/processed/y_test.csv', index=False)

print("✅ Feature-engineered datasets saved successfully!")
print("   • dataset/processed/X_train_featured.csv")
print("   • dataset/processed/X_test_featured.csv")
print("   • dataset/processed/y_train.csv")
print("   • dataset/processed/y_test.csv")

✅ Feature-engineered datasets saved successfully!
   • dataset/processed/X_train_featured.csv
   • dataset/processed/X_test_featured.csv
   • dataset/processed/y_train.csv
   • dataset/processed/y_test.csv
